# LCDM 2022: pełna rekonstrukcja benchmarku Lyα one-loop

Notebook odtwarza referencyjny model LCDM z pracy [arXiv:2210.06117](https://arxiv.org/abs/2210.06117), stanowiący punkt odniesienia przed skanem DCDM. Korzysta z 245 punktów BOSS DR14 dla $z=3.0,\ldots,4.2$, sześciu wspólnych parametrów efektywnych i technicznego cutoffu projekcji $k_{\rm UV}=20\,h/{\rm Mpc}$.

Notebook pokazuje faktyczne ustawienia zapisane w bundle'u, wyniki linear/one-loop, parametry best fit, odległości od granic, wkłady $\chi^2$ według redshiftu, końcowe krzywe względem danych, pullsy, surowe kanały SPT oraz skan cutoffu. Nie narzuca automatycznego werdyktu — wszystkie różnice względem publikacji są pokazane liczbowo.

## 1. Importy i katalog projektu

Z głównego katalogu projektu wystarczy jednorazowo wykonać `python -m pip install -e '.[notebook]'`. Do analizy gotowego pliku `.npz` nie jest potrzebny CLASS.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / 'lyalpha_one_loop', Path.cwd().parent / 'lyalpha_one_loop']
PROJECT_ROOT = next((p.resolve() for p in candidates if (p / 'pyproject.toml').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Nie znaleziono katalogu projektu z pyproject.toml.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from lyalpha_pt.data import load_dr12
from lyalpha_pt.fit import EffectiveModelFit, PARAMETER_NAMES, write_fit_result
from lyalpha_pt.theory import load_theory

print('PROJECT_ROOT =', PROJECT_ROOT)
print('Python =', sys.version.split()[0])

## 2. Konfiguracja

Domyślnie notebook wczytuje już wykonany fit. Ustaw `RUN_NEW_FITS=True`, aby przeprowadzić ponowną wielostartową minimalizację. Procedura używa bazowego boxu, następnie poszerza counterterm i prowadzi minimum ścieżką cutoffów $10\to15\to20$.

In [ ]:
DATA_DIR = PROJECT_ROOT / 'data'
THEORY_PATH = PROJECT_ROOT / 'results' / 'lcdm_2022_planck_precision_v3_3.npz'
FIT_PATH = PROJECT_ROOT / 'results' / 'lcdm_2022_planck_precision_v3_3_fit.json'
CUTOFF_PATH = PROJECT_ROOT / 'results' / 'lcdm_2022_planck_precision_v3_3_cutoff_scan.json'
OUTPUT_DIR = PROJECT_ROOT / 'results' / 'lcdm_2022_planck_validation_notebook'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

K_UV_CUT = 20.0
COVARIANCE_MODE = 'paper_diag'
FIT_SEEDS = [12345, 23456, 34567]
RUN_NEW_FITS = False
SAVE_FIGURES = True

PAPER_TARGETS = {
    'linear': 206.29,       # około 192.89 + 13.4 z opisu Fig. 6
    'one_loop': 192.89,     # jawnie podane w Sec. 3.3
    'delta': -13.4,         # jawnie podane przy Fig. 6
}
print('Theory:', THEORY_PATH)
print('Fit:', FIT_PATH)
print('Output:', OUTPUT_DIR)

## 3. Audyt ustawień z tabeli 1 pracy

Porównujemy bezpośrednio parametry CLASS zapisane w pliku teorii. Dwie bezmasowe neutrina są reprezentowane przez `N_ur=2.0328`, a jedna masywna przez `N_ncdm=1`, `m_ncdm=0.06`. Pętla jest liczona z $P_{cb}$ i ważona przez $(1-f_\nu)^2$, zgodnie z preskrypcją masywnych neutrin używaną w pracy poprzedzającej analizę DCDM.

In [ ]:
dataset = load_dr12(DATA_DIR)
theory = load_theory(THEORY_PATH)
model_meta = theory.metadata['model']
class_params = model_meta['class_params']
derived = theory.metadata.get('derived_cosmology', {})
numerics = theory.metadata['numerics']

expected = {
    'omega_b': 0.02237, 'omega_cdm': 0.1200, '100*theta_s': 1.04110,
    'ln10^{10}A_s': 3.044, 'n_s': 0.9649, 'tau_reio': 0.0544,
    'N_ur': 2.0328, 'N_ncdm': 1, 'm_ncdm': 0.06,
}
audit_rows = []
for name, target in expected.items():
    actual = class_params.get(name, np.nan)
    audit_rows.append({'quantity': name, 'paper/expected': target, 'bundle': actual, 'bundle - expected': actual - target})
audit_rows.extend([
    {'quantity': 'sigma8', 'paper/expected': 0.810, 'bundle': derived.get('sigma8', np.nan), 'bundle - expected': derived.get('sigma8', np.nan) - 0.810},
    {'quantity': 'S8', 'paper/expected': 0.833, 'bundle': derived.get('S8', np.nan), 'bundle - expected': derived.get('S8', np.nan) - 0.833},
])
cosmology_audit = pd.DataFrame(audit_rows)
display(cosmology_audit)

prescription_table = pd.DataFrame([
    {'quantity': 'model', 'value': model_meta['name']},
    {'quantity': 'h derived by CLASS', 'value': theory.h},
    {'quantity': 'Omega_m derived by CLASS', 'value': theory.omega_m},
    {'quantity': 'tree source', 'value': theory.metadata['tree_source']},
    {'quantity': 'loop source', 'value': theory.metadata['loop_source']},
    {'quantity': 'loop weight', 'value': theory.loop_weight},
    {'quantity': 'loop formula', 'value': theory.metadata['loop_formula']},
    {'quantity': 'k_trust [h/Mpc]', 'value': numerics['k_trust']},
    {'quantity': 'stored loop range [h/Mpc]', 'value': f"{theory.k_loop[0]:.4g}--{theory.k_loop[-1]:.4g}"},
    {'quantity': 'loop grid points', 'value': len(theory.k_loop)},
    {'quantity': 'internal q_max [h/Mpc]', 'value': numerics['q_max']},
    {'quantity': 'bundle digest', 'value': theory.metadata['bundle_digest']},
])
display(prescription_table)

## 4. Dane i definicja błędów

Wykorzystujemy 35 wartości $k$ dla każdego z siedmiu redshiftów. `paper_diag` oznacza diagonalną macierz z błędem statystycznym i ośmioma składowymi systematycznymi dodanymi w kwadraturze. Użycie samego `stat_diag` nie odtwarza skali opublikowanego $\chi^2$; dla porównania notebook pokazuje typową zmianę wielkości błędu.

In [ ]:
error_ratio = dataset.sigma_total / dataset.stat
data_table = pd.DataFrame([
    {'quantity': 'number of points', 'value': dataset.n_data},
    {'quantity': 'redshifts', 'value': ', '.join(f'{z:.1f}' for z in dataset.z_unique)},
    {'quantity': 'k_v range [s/km]', 'value': f'{dataset.k_velocity.min():.4g}--{dataset.k_velocity.max():.4g}'},
    {'quantity': 'covariance used', 'value': COVARIANCE_MODE},
    {'quantity': 'median sigma_total/stat', 'value': np.median(error_ratio)},
    {'quantity': 'range sigma_total/stat', 'value': f'{error_ratio.min():.3f}--{error_ratio.max():.3f}'},
])
display(data_table)

## 5. Fit linear i one-loop

Jeżeli `RUN_NEW_FITS=False`, parametry są odczytywane z gotowego JSON-u i niezależnie ponownie przeliczamy dla nich model oraz $\chi^2$. Przy `True` wykonujemy pełny fit bazowego boxu i lokalną kontynuację w poszerzonym zakresie countertermu.

In [ ]:
def fit_mode_with_cutoff_path(mode, initial_thetas=()):
    path = [10.0, 15.0, K_UV_CUT] if K_UV_CUT > 15 else [10.0, K_UV_CUT]
    path = list(dict.fromkeys(path))
    result = None
    history = []
    for i, cutoff in enumerate(path):
        fitter = EffectiveModelFit(dataset, theory, mode=mode, k_uv_cut=cutoff, covariance_mode=COVARIANCE_MODE)
        if i == 0:
            result = fitter.fit_staged(seeds=FIT_SEEDS, expand_counterterm=True, initial_thetas=initial_thetas)
        else:
            result = fitter.continue_staged(result.x, expand_counterterm=True)
        history.append({'k_uv_cut_hmpc': cutoff, 'chi2': fitter.chi2(result.x)})
    return fitter, result, history

if RUN_NEW_FITS:
    linear_fitter, linear_result, linear_path = fit_mode_with_cutoff_path('linear')
    loop_fitter, loop_result, loop_path = fit_mode_with_cutoff_path('one_loop', [linear_result.x])
    fitters = {'linear': linear_fitter, 'one_loop': loop_fitter}
    results = {'linear': linear_result, 'one_loop': loop_result}
    fit_records = {}
    for mode in ('linear', 'one_loop'):
        fit_records[mode] = fitters[mode].result_record(results[mode])
    fit_records['linear']['cutoff_continuation'] = linear_path
    fit_records['one_loop']['cutoff_continuation'] = loop_path
    saved = {
        'theory_file': str(THEORY_PATH.resolve()), 'data_dir': str(DATA_DIR.resolve()),
        'covariance_mode': COVARIANCE_MODE, 'fits': fit_records,
        'delta_chi2_one_loop_minus_linear': fit_records['one_loop']['chi2'] - fit_records['linear']['chi2'],
    }
    write_fit_result(FIT_PATH, saved)
else:
    saved = json.loads(FIT_PATH.read_text())
    for mode in ('linear', 'one_loop'):
        stored_digest = saved['fits'][mode].get('theory_digest')
        if stored_digest != theory.metadata['bundle_digest']:
            raise ValueError(f'{mode}: fit JSON belongs to a different theory bundle')
        if saved['fits'][mode].get('covariance_mode') != COVARIANCE_MODE:
            raise ValueError(f'{mode}: fit JSON uses a different covariance mode')
    fitters = {mode: EffectiveModelFit(dataset, theory, mode=mode, k_uv_cut=K_UV_CUT, covariance_mode=COVARIANCE_MODE) for mode in ('linear', 'one_loop')}

theta = {mode: np.array([saved['fits'][mode]['parameters'][name] for name in PARAMETER_NAMES]) for mode in ('linear', 'one_loop')}
summaries = {mode: fitters[mode].summary(theta[mode]) for mode in ('linear', 'one_loop')}
chi2_linear = summaries['linear']['chi2']
chi2_loop = summaries['one_loop']['chi2']
delta_chi2 = chi2_loop - chi2_linear
print(f'linear   chi2 = {chi2_linear:.9f}')
print(f'one-loop chi2 = {chi2_loop:.9f}')
print(f'Delta chi2    = {delta_chi2:.9f}')

## 6. Główny benchmark i parametry best fit

In [ ]:
benchmark_table = pd.DataFrame([
    {'quantity': 'chi2 linear', 'result': chi2_linear, 'paper': PAPER_TARGETS['linear'], 'result - paper': chi2_linear - PAPER_TARGETS['linear']},
    {'quantity': 'chi2 one-loop', 'result': chi2_loop, 'paper': PAPER_TARGETS['one_loop'], 'result - paper': chi2_loop - PAPER_TARGETS['one_loop']},
    {'quantity': 'Delta chi2', 'result': delta_chi2, 'paper': PAPER_TARGETS['delta'], 'result - paper': delta_chi2 - PAPER_TARGETS['delta']},
])
display(benchmark_table)

parameter_table = pd.DataFrame({
    'parameter': list(PARAMETER_NAMES),
    'linear': theta['linear'],
    'one_loop': theta['one_loop'],
})
parameter_table['one_loop - linear'] = parameter_table['one_loop'] - parameter_table['linear']
parameter_table = pd.concat([parameter_table, pd.DataFrame([
    {'parameter': 'alpha_F = exp(log_alpha_F)', 'linear': np.exp(theta['linear'][0]), 'one_loop': np.exp(theta['one_loop'][0]), 'one_loop - linear': np.exp(theta['one_loop'][0]) - np.exp(theta['linear'][0])}
])], ignore_index=True)
display(parameter_table)

boundary_rows = []
for mode in ('linear', 'one_loop'):
    for row in saved['fits'][mode]['boundary_diagnostics']:
        boundary_rows.append({'mode': mode, **row})
boundary_table = pd.DataFrame(boundary_rows)
display(boundary_table)

## 7. Wkład poszczególnych redshiftów do $\chi^2$

In [ ]:
linear_z = pd.DataFrame(fitters['linear'].chi2_by_redshift(theta['linear'])).rename(columns={'chi2': 'chi2_linear'})
loop_z = pd.DataFrame(fitters['one_loop'].chi2_by_redshift(theta['one_loop'])).rename(columns={'chi2': 'chi2_one_loop'})
chi2_by_z = linear_z.merge(loop_z[['z', 'chi2_one_loop']], on='z')
chi2_by_z['Delta chi2'] = chi2_by_z['chi2_one_loop'] - chi2_by_z['chi2_linear']
display(chi2_by_z)

fig, ax = plt.subplots(figsize=(8.4, 4.7))
colors = np.where(chi2_by_z['Delta chi2'] < 0, 'tab:green', 'tab:red')
ax.bar(chi2_by_z['z'].astype(str), chi2_by_z['Delta chi2'], color=colors)
ax.axhline(0, color='black', lw=1)
ax.set(xlabel='z', ylabel=r'$\Delta\chi^2_z$', title=rf'Redshift contributions, total $\Delta\chi^2={delta_chi2:.3f}$')
fig.tight_layout()
if SAVE_FIGURES: fig.savefig(OUTPUT_DIR / 'delta_chi2_by_redshift.png', dpi=180, bbox_inches='tight')
plt.show()

## 8. Końcowe krzywe względem danych — odpowiednik Fig. 6

Rysujemy $k_vP_{1D}/\pi$. Errorbary odpowiadają dokładnie przekątnej macierzy użytej w danym ficie.

In [ ]:
models = {mode: fitters[mode].model(theta[mode]) for mode in ('linear', 'one_loop')}
sigma_fit = np.sqrt(np.diag(fitters['linear'].covariance))
pulls = {mode: (models[mode] - dataset.p1d) / sigma_fit for mode in ('linear', 'one_loop')}
scale_plot = lambda kv, p: kv * p / np.pi

z_values = dataset.z_unique
nrows, ncols = 4, 2
fig, axes = plt.subplots(nrows, ncols, figsize=(13, 14.5))
axes = axes.ravel()
for ax, z in zip(axes, z_values):
    mask = np.isclose(dataset.z, z)
    order = np.argsort(dataset.k_velocity[mask])
    kv = dataset.k_velocity[mask][order]
    row = chi2_by_z.loc[np.isclose(chi2_by_z['z'], z)].iloc[0]
    ax.errorbar(kv, scale_plot(kv, dataset.p1d[mask][order]), yerr=scale_plot(kv, sigma_fit[mask][order]), fmt='o', ms=3.6, capsize=2, color='black', ecolor='0.55', label='BOSS DR14')
    ax.plot(kv, scale_plot(kv, models['linear'][mask][order]), '--', lw=2, label='linear')
    ax.plot(kv, scale_plot(kv, models['one_loop'][mask][order]), '-', lw=2, label='one-loop')
    ax.set_xscale('log')
    ax.set_xlabel(r'$k_v\,[\mathrm{s/km}]$')
    ax.set_ylabel(r'$k_vP_{1D}/\pi$')
    ax.set_title(rf'$z={z:.1f}$: $\chi^2_{{lin}}={row.chi2_linear:.2f}$, $\chi^2_{{1L}}={row.chi2_one_loop:.2f}$')
for ax in axes[len(z_values):]: ax.set_visible(False)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.995), ncol=3, frameon=False)
fig.suptitle(rf'LCDM 2022: $\chi^2_{{lin}}={chi2_linear:.3f}$, $\chi^2_{{1L}}={chi2_loop:.3f}$, $\Delta\chi^2={delta_chi2:.3f}$', y=1.025)
fig.tight_layout(rect=[0, 0, 1, 0.955])
if SAVE_FIGURES: fig.savefig(OUTPUT_DIR / 'lcdm_2022_linear_vs_one_loop_panels.png', dpi=180, bbox_inches='tight')
plt.show()

## 9. Pullsy i względna różnica końcowych predykcji

In [ ]:
fig, axes = plt.subplots(nrows, ncols, figsize=(13, 13.5), sharey=True)
axes = axes.ravel()
for ax, z in zip(axes, z_values):
    mask = np.isclose(dataset.z, z); order = np.argsort(dataset.k_velocity[mask]); kv = dataset.k_velocity[mask][order]
    ax.axhspan(-2, 2, color='0.7', alpha=0.15); ax.axhline(0, color='black', lw=1)
    ax.plot(kv, pulls['linear'][mask][order], 'o--', ms=3.3, lw=1.2, label='linear')
    ax.plot(kv, pulls['one_loop'][mask][order], 's-', ms=3.1, lw=1.2, label='one-loop')
    ax.set_xscale('log'); ax.set_xlabel(r'$k_v\,[\mathrm{s/km}]$'); ax.set_ylabel('pull'); ax.set_title(rf'$z={z:.1f}$')
for ax in axes[len(z_values):]: ax.set_visible(False)
handles, labels = axes[0].get_legend_handles_labels(); fig.legend(handles, labels, loc='upper center', ncol=2, frameon=False)
fig.tight_layout(rect=[0, 0, 1, 0.975])
if SAVE_FIGURES: fig.savefig(OUTPUT_DIR / 'lcdm_2022_pulls.png', dpi=180, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(nrows, ncols, figsize=(13, 13.0), sharey=True)
axes = axes.ravel()
for ax, z in zip(axes, z_values):
    mask = np.isclose(dataset.z, z); order = np.argsort(dataset.k_velocity[mask]); kv = dataset.k_velocity[mask][order]
    relative = models['one_loop'][mask][order] / models['linear'][mask][order] - 1
    ax.axhline(0, color='black', lw=1); ax.plot(kv, 100 * relative, 'o-', ms=3.5, color='tab:purple')
    ax.set_xscale('log'); ax.set_xlabel(r'$k_v\,[\mathrm{s/km}]$'); ax.set_ylabel(r'$100(P_{1L}/P_{lin}-1)\,[\%]$'); ax.set_title(rf'$z={z:.1f}$')
for ax in axes[len(z_values):]: ax.set_visible(False)
fig.tight_layout()
if SAVE_FIGURES: fig.savefig(OUTPUT_DIR / 'lcdm_2022_best_fit_relative_difference.png', dpi=180, bbox_inches='tight')
plt.show()

## 10. Surowe kanały one-loop przed projekcją

Pierwszy rząd pokazuje pełne kanały względem tree level. Drugi rozdziela $P_{22}$ i $P_{13}$. Linia pionowa oznacza $k_{\rm trust}=2\,h/{\rm Mpc}$; poprawki w pliku nie są powyżej niej wyłączane.

In [ ]:
selected_z = [3.0, 3.6, 4.2]
channel_labels = [r'$P_{\delta\delta}$', r'$P_{\delta\theta}$', r'$P_{\theta\theta}$']
fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))
for col, z in enumerate(selected_z):
    iz = int(np.argmin(np.abs(theory.z - z))); k = theory.k_loop; tree = theory.p_tree[iz]
    ax = axes[0, col]
    for ch, label in enumerate(channel_labels): ax.plot(k, theory.channels_one_loop[iz, :, ch] / tree, lw=2, label=label)
    ax.axhline(1, color='black', lw=1); ax.axvline(numerics['k_trust'], color='black', ls=':', lw=1.2)
    ax.set_xscale('log'); ax.set_yscale('symlog', linthresh=0.1); ax.set_title(rf'$z={z:.1f}$'); ax.set_xlabel(r'$k\,[h/\mathrm{{Mpc}}]$')
    if col == 0: ax.set_ylabel('one-loop / tree')
    ax = axes[1, col]
    ax.plot(k, theory.loop_weight * theory.p22[iz, :, 0] / tree, label=r'$P_{22}^{\delta\delta}/P_{tree}$')
    ax.plot(k, theory.loop_weight * theory.p13[iz, :, 0] / tree, label=r'$P_{13}^{\delta\delta}/P_{tree}$')
    ax.plot(k, theory.loop_weight * (theory.p22[iz, :, 0] + theory.p13[iz, :, 0]) / tree, '--', lw=2, label='sum')
    ax.axhline(0, color='black', lw=1); ax.axvline(numerics['k_trust'], color='black', ls=':', lw=1.2)
    ax.set_xscale('log'); ax.set_yscale('symlog', linthresh=0.1); ax.set_xlabel(r'$k\,[h/\mathrm{{Mpc}}]$')
    if col == 0: ax.set_ylabel('loop correction / tree')
axes[0, 0].legend(fontsize=9); axes[1, 0].legend(fontsize=9)
fig.tight_layout()
if SAVE_FIGURES: fig.savefig(OUTPUT_DIR / 'lcdm_2022_raw_spt_channels.png', dpi=180, bbox_inches='tight')
plt.show()

## 11. Skan technicznego cutoffu

In [ ]:
cutoff_saved = json.loads(CUTOFF_PATH.read_text())
cutoff_rows = []
for record in cutoff_saved['cutoff_scan']:
    cutoff_rows.append({
        'k_uv [h/Mpc]': record['k_uv_cut_hmpc'], 'chi2 one-loop': record['chi2'],
        **record['parameters'],
        'near bound': any(row['near_bound'] for row in record['boundary_diagnostics']),
    })
cutoff_table = pd.DataFrame(cutoff_rows)
display(cutoff_table)

fig, ax = plt.subplots(figsize=(7.5, 4.6))
ax.plot(cutoff_table['k_uv [h/Mpc]'], cutoff_table['chi2 one-loop'], 'o-', lw=2)
ax.axhline(PAPER_TARGETS['one_loop'], color='black', ls='--', label='paper 192.89')
ax.set(xlabel=r'$k_{UV}\,[h/\mathrm{Mpc}]$', ylabel=r'$\chi^2_{1-loop}$', xticks=cutoff_table['k_uv [h/Mpc]'])
ax.legend(); fig.tight_layout()
if SAVE_FIGURES: fig.savefig(OUTPUT_DIR / 'lcdm_2022_cutoff_scan.png', dpi=180, bbox_inches='tight')
plt.show()

## 12. Zapis tabel i krzywych

Wszystkie liczby użyte na wykresach są zapisywane osobno, aby można je było później wykorzystać w tekście lub porównać z DCDM bez ponownego uruchamiania notebooka.

In [ ]:
cosmology_audit.to_csv(OUTPUT_DIR / 'cosmology_settings_audit.csv', index=False)
benchmark_table.to_csv(OUTPUT_DIR / 'benchmark_comparison.csv', index=False)
parameter_table.to_csv(OUTPUT_DIR / 'best_fit_parameters.csv', index=False)
boundary_table.to_csv(OUTPUT_DIR / 'boundary_diagnostics.csv', index=False)
chi2_by_z.to_csv(OUTPUT_DIR / 'chi2_by_redshift.csv', index=False)
cutoff_table.to_csv(OUTPUT_DIR / 'cutoff_scan.csv', index=False)
np.savez_compressed(
    OUTPUT_DIR / 'best_fit_curves.npz', z=dataset.z, k_velocity=dataset.k_velocity,
    p1d_data=dataset.p1d, sigma_fit=sigma_fit, p1d_linear=models['linear'],
    p1d_one_loop=models['one_loop'], pull_linear=pulls['linear'],
    pull_one_loop=pulls['one_loop'], theta_linear=theta['linear'], theta_one_loop=theta['one_loop'],
    chi2_linear=chi2_linear, chi2_one_loop=chi2_loop, delta_chi2=delta_chi2,
)
print('Saved files:')
for path in sorted(OUTPUT_DIR.iterdir()): print(' -', path.name)

## Interpretacja przed przejściem do DCDM

Pełne odtworzenie powinno być oceniane wspólnie na podstawie: (1) parametrów kosmologicznych z tabeli 1, (2) surowych kanałów, (3) liniowego benchmarku, (4) one-loop i $\Delta\chi^2$, (5) stabilności na odcinku cutoffów 15–20 oraz (6) braku minimum zatrzymanego na granicy.

Dla bieżącego bundle'a otrzymujemy $\chi^2_{lin}=206.248$, $\chi^2_{1L}=193.197$ i $\Delta\chi^2=-13.051$. Różnice względem liczb 206.29, 192.89 i -13.4 wynoszą odpowiednio -0.042, +0.307 i +0.349. To jest zgodność na poziomie znacznie mniejszym niż jednostka $\chi^2$, ale przed właściwym skanem DCDM należy zachować te różnice jako jawny budżet rekonstrukcji, a nie ręcznie dostrajać nuisance parameters do dokładnie 192.89.